In [ ]:
import polars as pl
from pathlib import Path

DATA = Path("../data/processed/yelp_reviews_with_embeddings.parquet")

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
token_limit = tokenizer.model_max_length

In [ ]:
df = pl.read_parquet(DATA)

In [ ]:

df = df.with_columns(
    pl.col("text")
    .map_elements(
        lambda s: len(tokenizer.encode(s, add_special_tokens=True)),
        return_dtype=pl.Int64
    ).alias("token_count")
)
df.head()

In [ ]:
exceeded_df = df.filter(pl.col("token_count") > token_limit)
exceeded_df

In [ ]:
lf = pl.scan_parquet(DATA)
lf.head().collect()

In [ ]:
lf.select("user_review_count").describe()

In [ ]:
lf.select("user_average_stars").describe()

In [ ]:
lf.select("business_review_count").describe()

In [ ]:
lf_transformed = lf.with_columns(
    pl.col("user_review_count").log().alias("log_user_review_count"),
    pl.col("business_review_count").log().alias("log_business_review_count"),
    pl.col("state").cast(pl.Categorical)
)
lf_transformed.head().collect()

In [ ]:
lf_transformed.sink_parquet("../data/processed/yelp_with_embeddings_processed.parquet")